<a href="https://colab.research.google.com/github/xpdlaldam/decoder_only_transformers/blob/main/decoder_only_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 6.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 13.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 16.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [lightning]


In [2]:
import torch
import torch.nn as nn # to use nn.Module(), nn.Embedding(), nn.Linear()
import torch.nn.functional as F # to use softmax(), argmax()
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader # to store our data in DataLoaders
import lightning as L # makes it easier to write, optimize, scale our code

In [3]:
# first, create a dictionary that maps vocabulary tokens to id numbers
# this will be used on the way in
# vocabulary: word -> id
token_to_id = {
    'is'        : 0,
    'charizard' : 1,
    'cool'      : 2,
    'yes'       : 3,
    '<EOS>'     : 4,
}

# then we create a dictionary that maps the ids to tokens which will help us interpret the output
# use the map() function to apply the reversed() function to each tuple (i.e. ('is', 0)) stored in the token_to_id dictionary
# then use dict() to make a new dictionary from the reversed tuples
# this will be used on the way out
# reverse lookup: id -> word (for reading the model's output)
id_to_token = dict(map(reversed, token_to_id.items()))
id_to_token

# inputs: question + <EOS> + answer (decoder-only, teacher forcing)
inputs = torch.tensor([[token_to_id["is"], # input 1: is charizard cool <EOS> yes
                        token_to_id["charizard"],
                        token_to_id["cool"],
                        token_to_id["<EOS>"],
                        token_to_id["yes"]],

                       [token_to_id["charizard"], # input 2: charizard is cool <EOS> yes
                        token_to_id["is"],
                        token_to_id["cool"],
                        token_to_id["<EOS>"],
                        token_to_id["yes"]]])

## labels: the inputs shifted one to the left, ending with <EOS>
## each position's label is "the token that should come next"
labels = torch.tensor([[token_to_id["charizard"],
                        token_to_id["cool"],
                        token_to_id["<EOS>"],
                        token_to_id["yes"],
                        token_to_id["<EOS>"]],

                       [token_to_id["is"],
                        token_to_id["cool"],
                        token_to_id["<EOS>"],
                        token_to_id["yes"],
                        token_to_id["<EOS>"]]])

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)